# Week 5 AI/ML Assignment: Logistic Regression for Crack Detection

**Objective:** Build and analyze a Logistic Regression model to predict crack presence in mechanical components based on Stress, Stress Intensity (K), and Load Cycles.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, roc_curve, auc

sns.set(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Data Loading and Exploration

In [2]:
df = pd.read_csv("../data/CrackDetectiondataset.csv", usecols=[0, 1, 2, 3])
df.columns = ['Stress', 'K', 'Cycles', 'Crack']

print(f"Dataset Statistics:\n{df.describe()}")
print(f"\nClass Distribution:\n{df['Crack'].value_counts()}")
df.head()

Dataset Statistics:
           Stress           K      Cycles       Crack
count  100.000000  100.000000  100.000000  100.000000
mean   438.072400   39.695300  168.324800    0.070000
std    118.995357   12.155702   86.299105    0.256432
min    252.210000   20.280000   21.420000    0.000000
max    644.750000   59.430000  297.220000    1.000000

Class Distribution:
0    93
1     7
Name: Crack, dtype: int64


## 2. Model Training (Standardized)

In [3]:
X = df[['Stress', 'K', 'Cycles']]
y = df['Crack']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LogisticRegression(random_state=42)
model.fit(X_scaled, y)

print("Model trained on standardized features.")

Model trained on standardized features.


## 3. Extracting and Comparing Coefficients

In [4]:
weights = model.coef_[0]
intercept = model.intercept_[0]
means = scaler.mean_
scales = scaler.scale_

unscaled_beta_1, unscaled_beta_2, unscaled_beta_3 = weights / scales
unscaled_beta_0 = intercept - np.sum((weights * means) / scales)

comparison = pd.DataFrame({
    'Parameter': ['Intercept (B0)', 'Stress (B1)', 'K (B2)', 'Cycles (B3)'],
    'Hand_Calculated': [-15.0, 0.015, 0.08, 0.01],
    'ML_Derived': [unscaled_beta_0, unscaled_beta_1, unscaled_beta_2, unscaled_beta_3]
})
comparison

         Parameter  Hand_Calculated  ML_Derived
0   Intercept (B0)           -15.00  -19.632128
1      Stress (B1)             0.015    0.016391
2           K (B2)             0.08    0.149429
3      Cycles (B3)             0.01    0.011406

## 4. Threshold and Ratio Analysis

In [5]:
y_probs = model.predict_proba(X_scaled)[:, 1]
thresholds = np.arange(0.1, 1.0, 0.1)
analysis = []

for t in thresholds:
    y_pred = (y_probs >= t).astype(int)
    yes = np.sum(y_pred)
    no = len(y_pred) - yes
    analysis.append({
        'Threshold': round(t, 1),
        'Accuracy': accuracy_score(y, y_pred),
        'Precision': precision_score(y, y_pred, zero_division=0),
        'Recall': recall_score(y, y_pred, zero_division=0),
        'Ratio_Yes_No': yes/no if no > 0 else np.inf
    })

pd.DataFrame(analysis)

   Threshold  Accuracy  Precision    Recall  Ratio_Yes_No
0        0.1      0.88   0.368421  1.000000      0.234568
1        0.2      0.97   0.700000  1.000000      0.111111
2        0.3      1.00   1.000000  1.000000      0.075269
3        0.4      0.98   1.000000  0.714286      0.052632
4        0.5      0.98   1.000000  0.714286      0.052632
5        0.6      0.96   1.000000  0.428571      0.030928
6        0.7      0.93   0.000000  0.000000      0.000000
7        0.8      0.93   0.000000  0.000000      0.000000
8        0.9      0.93   0.000000  0.000000      0.000000

## 5. Visualizations

In [6]:
# Performance Curves
res_df = pd.DataFrame(analysis)
plt.plot(res_df['Threshold'], res_df['Accuracy'], 'o-', label='Accuracy')
plt.plot(res_df['Threshold'], res_df['Precision'], 's-', label='Precision')
plt.plot(res_df['Threshold'], res_df['Recall'], '^-', label='Recall')
plt.legend()
plt.title("Metrics vs Threshold")
plt.show()

## 6. Discussion and Interpretation

1. **Threshold Selection:** A low threshold (0.1-0.2) ensures all cracks are caught (High Recall) but increases false alarms. A high threshold (0.7+) makes the model ignore safety risks.
2. **Safety vs. Cost:** In engineering, we favor Recall to avoid structural failure, even if it means higher inspection costs due to false positives.
3. **Model Significance:** The ML model shows that Stress Intensity (K) is more influential than the hand-calculated baseline originally suggested.